In [1]:
import torch
import gradio as gr
import json
import numpy as np

# 1. Загружаем словарь (укажи правильный путь к своему vocab.json)
with open('./gym_bert_data/vocab.json', 'r') as f:
    vocab = json.load(f)

# Обратный словарь для декодирования (ID -> Название)
id2token = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

# 2. Определяем архитектуру (должна быть 1-в-1 как при обучении)
class BERT4Rec(torch.nn.Module):
    def __init__(self, vocab_size, hidden=256, heads=8, layers=4, max_len=20):
        super().__init__()
        self.token_emb = torch.nn.Embedding(vocab_size, hidden)
        self.pos_emb = torch.nn.Embedding(max_len, hidden)
        encoder_layer = torch.nn.TransformerEncoderLayer(d_model=hidden, nhead=heads, batch_first=True)
        self.transformer = torch.nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.fc = torch.nn.Linear(hidden, vocab_size)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.token_emb(x) + self.pos_emb(positions)
        x = self.transformer(x)
        return self.fc(x)

# 3. Инициализация и загрузка весов
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = BERT4Rec(vocab_size).to(device)
model.load_state_dict(torch.load('./gym_bert_data/checkpoints-2/gym_bert_v2_ep27_hit0.3522.pth', map_location=device))
model.eval()

# 4. Функция предсказания
def predict_next_exercises(exercise_list_str):
    # Превращаем строку в список (через запятую)
    input_names = [ex.strip() for ex in exercise_list_str.split(',') if ex.strip()]
    
    # Переводим названия в ID (если слова нет в словаре - пропускаем)
    input_ids = [vocab[name] for name in input_names if name in vocab]
    
    if not input_ids:
        return "Упражнения не найдены в словаре. Проверь написание!"

    # Подготовка тензора (добавляем маску [MASK] в конец, если это нужно модели)
    # В твоем случае модель учится на всей последовательности, берем последний выход
    input_tensor = torch.LongTensor([input_ids]).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        # Берем вероятности только для самого последнего токена в цепочке
        last_token_logits = output[0, -1, :] 
        probabilities = torch.softmax(last_token_logits, dim=0)
        
        # Берем Топ-5 предсказаний
        top5_probs, top5_inds = torch.topk(probabilities, 5)
        
    result = ""
    for i in range(5):
        ex_name = id2token[top5_inds[i].item()]
        conf = top5_probs[i].item() * 100
        result += f"{i+1}. **{ex_name}** ({conf:.2f}%)\n"
    
    return result

# 5. Создаем интерфейс Gradio
demo = gr.Interface(
    fn=predict_next_exercises,
    inputs=gr.Textbox(
        label="Введи упражнения через запятую", 
        placeholder="Bench Press (Barbell), Incline Dumbbell Press",
        lines=2
    ),
    outputs=gr.Markdown(label="Топ-5 рекомендаций от BERT"),
    title="🏋️‍♂️ Gym BERT Recommender (v1)",
    description="Введите список выполненных упражнений, и 4-слойный трансформер предскажет следующее.",
    examples=[
        
        # --- Грудь и Трицепс ---
        ["Bench Press (Barbell), Incline Dumbbell Press, Chest Fly (Cable)"],
        ["Push-Up, Dips (Chest Version), Triceps Extension (Cable)"],
        ["Dumbbell Fly, Low-To-High Cable Fly, Triceps Pushdown (Rope)"],
        
        # --- Спина и Бицепс ---
        ["Deadlift (Barbell), Pull-Up (Weighted), Seated Cable Row"],
        ["Lat Pulldown (Wide Grip), Bent Over Row (Barbell), Face Pull"],
        ["One-Arm Dumbbell Row, T-Bar Row, Biceps Curl (Dumbbell)"],
        ["Pull-Up, Lat Pulldown (Close Grip), Hammer Curl"],
        
        # --- Ноги (Квадрицепс / Бицепс бедра) ---
        ["Squat (Barbell), Leg Press, Leg Extension"],
        ["Bulgarian Split Squat, Walking Lunge, Leg Press"],
        ["Romanian Deadlift (Barbell), Leg Curl (Seated), Calf Raise"],
        ["Hack Squat, Leg Extension, Goblet Squat"],
        
        # --- Плечи ---
        ["Overhead Press (Barbell), Side Lateral Raise, Front Raise"],
        ["Military Press, Arnold Press, Reverse Fly (Dumbbell)"],
        ["Dumbbell Shoulder Press, Lateral Raise (Cable), Shrug (Barbell)"],
        
        # --- Смешанные / Фулбоди ---
        ["Burpee, Kettlebell Swing, Thruster"],
        ["Plank, Hanging Leg Raise, Russian Twist"],
        ["Clean and Jerk, Snatch, Front Squat"],
        
        # --- Короткие сессии (начало) ---
        ["Bench Press (Barbell)"],
        ["Squat (Barbell)"],
        ["Deadlift (Barbell)"]
    ]
)

if __name__ == "__main__":
    demo.launch(share=True)

/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 3.41.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://cab18949d59cf21e83.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [1]:
import torch
import gradio as gr
import json
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

# 1. Загрузка данных и словарей
with open('./gym_bert_data/vocab.json', 'r') as f:
    vocab = json.load(f)

id2token = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

# Загружаем онтологию для извлечения групп мышц (is_same_group)
# Предполагаем, что exercise_ontology уже создана в памяти или загружена
# (Если нет, она создается из df, как мы делали ранее)

# 2. Архитектура BERT4Rec
class BERT4Rec(torch.nn.Module):
    def __init__(self, vocab_size, hidden=256, heads=8, layers=4, max_len=20):
        super().__init__()
        self.token_emb = torch.nn.Embedding(vocab_size, hidden)
        self.pos_emb = torch.nn.Embedding(max_len, hidden)
        encoder_layer = torch.nn.TransformerEncoderLayer(d_model=hidden, nhead=heads, batch_first=True)
        self.transformer = torch.nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.fc = torch.nn.Linear(hidden, vocab_size)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.token_emb(x) + self.pos_emb(positions)
        x = self.transformer(x)
        return self.fc(x)

# 3. Инициализация моделей
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Загружаем BERT
bert_model = BERT4Rec(vocab_size).to(device)
bert_model.load_state_dict(torch.load('./gym_bert_data/checkpoints-2/gym_bert_v2_ep27_hit0.3522.pth', map_location=device))
bert_model.eval()

# Загружаем CatBoost (укажи путь, если сохранял модель отдельно)
# cat_ranker.load_model('cat_ranker.cbm') 
# Для теста используем текущую переменную cat_ranker из памяти

# 4. Функция гибридного предсказания
def hybrid_predict(exercise_list_str, goal, level, equipment):
    input_names = [ex.strip() for ex in exercise_list_str.split(',') if ex.strip()]
    input_ids = [vocab[name] for name in input_names if name in vocab]
    
    if not input_ids:
        return "❌ Упражнения не найдены в словаре."

    # --- STAGE 1: BERT RETRIEVAL ---
    input_tensor = torch.LongTensor([input_ids[-20:]]).to(device)
    with torch.no_grad():
        output = bert_model(input_tensor)
        probabilities = torch.softmax(output[0, -1, :], dim=0)
        # Берем топ-50 кандидатов для ранжирования
        top_probs, top_indices = torch.topk(probabilities, 50)
    
    # --- STAGE 2: CATBOOST RANKING ---
    last_ex_name = input_names[-1]
    last_muscle = exercise_ontology.get(last_ex_name, {}).get('level', 'None')
    
    candidates_data = []
    for prob, idx in zip(top_probs.cpu().numpy(), top_indices.cpu().numpy()):
        cand_name = id2token[idx]
        meta = exercise_ontology.get(cand_name, {})
        
        candidates_data.append({
            'bert_score': float(prob),
            'candidate_id': int(idx),
            'level': level,
            'goal': goal,
            'equipment': equipment,
            'is_same_group': 1 if meta.get('level') == last_muscle else 0
        })
    
    candidates_df = pd.DataFrame(candidates_data)
    
    # Получаем вероятности от CatBoost
    final_scores = cat_ranker.predict_proba(candidates_df)[:, 1]
    candidates_df['final_score'] = final_scores
    
    # Сортируем по результату CatBoost
    top_results = candidates_df.sort_values('final_score', ascending=False).head(5)
    
    result_md = f"### 🎯 Рекомендации для {goal} ({level})\n"
    for i, (idx, row) in enumerate(top_results.iterrows()):
        ex_name = id2token[int(row['candidate_id'])]
        match_str = "✅" if row['is_same_group'] else "🔄"
        result_md += f"{i+1}. **{ex_name}** | Скор: `{row['final_score']:.4f}` {match_str}\n"
    
    return result_md

# 5. Интерфейс Gradio
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏋️‍♂️ Smart Gym Hybrid Recommender")
    gr.Markdown("Система **BERT4Rec + CatBoost**. Нейронка предлагает варианты, бустинг фильтрует их под твой профиль.")
    
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="История тренировки (через запятую)", 
                placeholder="Bench Press (Barbell), Incline Dumbbell Press",
                value="Bench Press (Barbell), Incline Dumbbell Press"
            )
            user_goal = gr.Dropdown(
                choices=["Strength", "Hypertrophy", "Weight Loss", "Fitness"], 
                label="Цель тренировки", value="Strength"
            )
            user_level = gr.Dropdown(
                choices=["Beginner", "Intermediate", "Advanced"], 
                label="Ваш уровень", value="Intermediate"
            )
            user_equip = gr.Dropdown(
                choices=["Barbell", "Dumbbell", "Machine", "Bodyweight"], 
                label="Доступное оборудование", value="Barbell"
            )
            btn = gr.Button("Получить план", variant="primary")
            
        with gr.Column():
            output_md = gr.Markdown(label="Рекомендации")

    btn.click(
        fn=hybrid_predict, 
        inputs=[input_text, user_goal, user_level, user_equip], 
        outputs=output_md
    )
    
    gr.Examples(
        examples=[
            ["Squat (Barbell), Leg Press", "Strength", "Advanced", "Barbell"],
            ["Push-Up, Plank", "Weight Loss", "Beginner", "Bodyweight"],
            ["Deadlift (Barbell), Pull-Up (Weighted)", "Hypertrophy", "Advanced", "Barbell"]
        ],
        inputs=[input_text, user_goal, user_level, user_equip]
    )

if __name__ == "__main__":
    demo.launch(share=True)

/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 3.41.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://dd0419519102ca65b2.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/gradio/routes.py", line 488, in run_predict
    output = await app.get_blocks().process_api(
  File "/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/gradio/blocks.py", line 1431, in process_api
    result = await self.call_function(
  File "/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/gradio/blocks.py", line 1103, in call_function
    prediction = await anyio.to_thread.run_sync(
  File "/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
  File "/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/anyio/_backends/_asyncio.py", line 2502, in run_sync_in_worker_thread
    return await future
  File "/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/anyio/_backends/_asyncio.py", line 986, in run
    result = contex